# 11쪽 · Agent: 도구 결과를 보고 답하기

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 준비와 도구 함수

함수는 여기에서 직접 확인할 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition


def add_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 더합니다."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 곱합니다."""
    return a * b

## 2. 모델에 도구 연결

한 번의 응답에서 도구를 순차적으로 요청하도록 설정합니다.

In [ ]:
llm = make_llm()

tools = [add_numbers, multiply_numbers]
tool_llm = llm.bind_tools(
    tools,
    parallel_tool_calls=False,
)

## 3. Assistant 정의

도구 결과를 받은 뒤에도 같은 tool_llm을 사용합니다.

In [ ]:
system = SystemMessage(
    content=(
        "계산은 제공된 도구를 사용하세요. "
        "도구 결과를 확인하고 필요한 계산을 이어서 수행하세요. "
        "계산이 끝나면 한국어로 답하세요. "
        "이전 결과를 모르면 추측하지 말고 물어보세요."
    )
)


def assistant(state: MessagesState):
    messages = [system] + state["messages"]
    response = tool_llm.invoke(messages)

    return {"messages": [response]}

## 4. 그래프 연결

tools에서 assistant로 돌아오는 한 줄이 핵심입니다.

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)

# 도구 결과를 모델에게 돌려보냅니다.
builder.add_edge("tools", "assistant")

## 5. 그래프 준비

실행 가능한 그래프를 만듭니다.

In [ ]:
graph = builder.compile()

## 6. 덧셈 실행

도구 결과 다음에 모델의 최종 답변이 오는지 확인합니다.

In [ ]:
messages = [
    HumanMessage(content="10과 20을 더해줘. 덧셈 도구를 사용해줘."),
]

result = graph.invoke({"messages": messages})

for message in result["messages"]:
    message.pretty_print()

**결과 해설**

HumanMessage, AIMessage의 도구 요청, ToolMessage의 30, AIMessage의 최종 답변 순서를 확인합니다.